In [1]:
import os
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

DATA_PATH = os.getcwd() + "/data"


/vol/bitbucket/gk825/nlpenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

Torch: 2.2.2+cu118
CUDA available: True
GPU: NVIDIA GeForce GTX 1080


In [3]:
pcl_df = pd.read_csv(
    DATA_PATH + '/dontpatronizeme_pcl.tsv', 
    delimiter='\t', 
    header=None)
pcl_df.columns = ['paragraph_id', 'article_id', 'keyword', 'country_code', 'paragraph', 'label']
pcl_df["y"] = pcl_df["label"].apply(lambda x: 0 if x < 2 else 1)
pcl_df['paragraph'] = pcl_df['paragraph'].fillna("").astype(str)
pcl_df['paragraph_id'] = pcl_df['paragraph_id']


In [4]:
train_ids = pd.read_csv( DATA_PATH + '/train_semeval_parids-labels.csv')
test_ids = pd.read_csv(DATA_PATH + '/dev_semeval_parids-labels.csv')

train_df = pd.merge(pcl_df, train_ids, how='right', left_on='paragraph_id', right_on='par_id')
test_df = pd.merge(pcl_df, test_ids, how='right', left_on='paragraph_id', right_on='par_id')


In [5]:
pos_df = train_df[train_df.y == 1]
neg_df = train_df[train_df.y == 0]

npos = len(pos_df)

neg_df_down = neg_df.sample(n=min(npos * 2, len(neg_df)), random_state=42)

train_balanced = pd.concat([pos_df, neg_df_down]).sample(frac=1, random_state=42)

print(train_balanced.shape)

(2382, 9)


In [6]:
train_dataset = Dataset.from_pandas(train_balanced[["paragraph", "y"]])
dev_dataset = Dataset.from_pandas(test_df[["paragraph", "y"]])

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def tokenize(example):
    return tokenizer(
        example["paragraph"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset = train_dataset.map(tokenize, batched=True)
dev_dataset = dev_dataset.map(tokenize, batched=True)
train_dataset = train_dataset.rename_column("y", "labels")
dev_dataset = dev_dataset.rename_column("y", "labels")


train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
dev_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

/vol/bitbucket/gk825/nlpenv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Map: 100%|██████████| 2094/2094 [00:01<00:00, 2017.72 examples/s]


In [7]:
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,  # baseline = 1 epoch
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    evaluation_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="no",
    load_best_model_at_end=False,
    report_to="none"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    f1 = f1_score(labels, preds, pos_label=1)
    return {"f1_positive": f1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

metrics = trainer.evaluate()
print(metrics)

Epoch,Training Loss,Validation Loss,F1 Positive
1,0.494900,0.305888,0.491525


{'eval_loss': 0.3058875501155853, 'eval_f1_positive': 0.4915254237288136, 'eval_runtime': 32.7379, 'eval_samples_per_second': 63.963, 'eval_steps_per_second': 4.001, 'epoch': 1.0}
